# Course 2 — Feature Engineering

Scaling, polynomial expansions, encoding, interactions, and the
`Pipeline` pattern that holds it all together.

**Sections**
1. Scaling, skew, pipelines (0:00–0:30)
2. Polynomial features and the bias-variance picture (0:30–1:00)
3. Categorical encoding and interactions (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
rng = np.random.default_rng(0)
auto = load_dataset('auto')
tips = load_dataset('tips')
print(auto.shape, tips.shape)


## 1. Scaling, skew, pipelines

### Why scale at all?

In [ ]:
print('weight: min={:.0f}  max={:.0f}'.format(auto['weight'].min(), auto['weight'].max()))
print('accel : min={:.0f}  max={:.0f}'.format(auto['acceleration'].min(), auto['acceleration'].max()))


Linear regression doesn't care, but the moment you add regularization
(Ridge, Lasso in Course 4) or anything distance-based (KNN, SVM next
week), the column with the bigger numbers eats the model. Standardize.

In [ ]:
scaler = StandardScaler()
Z = scaler.fit_transform(auto[['weight', 'acceleration']])
print('after scaling mean/std:', Z.mean(axis=0).round(2), Z.std(axis=0).round(2))


### Skew and `log1p`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(auto['horsepower'], bins=30); axes[0].set_title('horsepower (raw)')
axes[1].hist(np.log1p(auto['horsepower']), bins=30); axes[1].set_title('log1p(horsepower)')
plt.show()


### Pipeline: scaler + linear model in one object

In [ ]:
pipe = Pipeline([('scale', StandardScaler()), ('lr', LinearRegression())])
X = auto[['weight', 'acceleration']]
y = auto['mpg']
pipe.fit(X, y)
print(f'R^2 = {pipe.score(X, y):.4f}')


## 2. Polynomial features

### Underfit, fit, overfit — the same data, four degrees

In [ ]:
x = auto[['horsepower']].to_numpy()
y = auto['mpg'].to_numpy()
order = np.argsort(x.ravel())
xs = x.ravel()[order]; ys = y[order]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(xs, ys, s=10, alpha=0.4, color='gray')
x_grid = np.linspace(xs.min(), xs.max(), 200).reshape(-1, 1)
for d, color in zip([1, 2, 5, 15], ['C0', 'C1', 'C2', 'C3']):
    poly = PolynomialFeatures(degree=d, include_bias=False)
    pipe = Pipeline([('p', poly), ('lr', LinearRegression())])
    pipe.fit(x, y)
    ax.plot(x_grid, pipe.predict(x_grid), label=f'd = {d}', color=color)
ax.set_xlabel('horsepower'); ax.set_ylabel('mpg'); ax.legend()
ax.set_title('Polynomial fits: d=1 underfit, d=15 overfit'); plt.show()


d = 1 is too straight, d = 15 starts wagging at the edges. Course 3
will give us a principled way to pick d via cross-validation.

## 3. Categorical encoding and interactions

### One-hot vs ordinal

In [ ]:
oh = OneHotEncoder(sparse_output=False, drop='first')
encoded = oh.fit_transform(tips[['day', 'time']])
print('OneHot columns:', oh.get_feature_names_out().tolist())
print(encoded[:3])


In [ ]:
# Ordinal encoding is appropriate only if categories have an order.
# 'size' (party size) is a count, but pretend day-of-week is ordered.
ord_enc = OrdinalEncoder(categories=[['Thur', 'Fri', 'Sat', 'Sun']])
print(ord_enc.fit_transform(tips[['day']])[:5].ravel())


### Interaction features

In [ ]:
X = pd.get_dummies(tips[['day', 'time', 'sex']], drop_first=True, dtype=float)
X['total_bill'] = tips['total_bill']
X['size'] = tips['size']
poly = PolynomialFeatures(interaction_only=True, include_bias=False)
X_int = poly.fit_transform(X)
print(f'before: {X.shape[1]} cols, after pairwise interactions: {X_int.shape[1]}')


### ColumnTransformer — different recipes for different columns

In [ ]:
num = ['total_bill', 'size']
cat = ['day', 'time', 'sex', 'smoker']
preproc = ColumnTransformer([
    ('num', StandardScaler(), num),
    ('cat', OneHotEncoder(drop='first'), cat),
])
pipe = Pipeline([('prep', preproc), ('lr', LinearRegression())])
pipe.fit(tips[num + cat], tips['tip'])
print(f'R^2 (tip ~ everything) = {pipe.score(tips[num + cat], tips["tip"]):.4f}')


### Recap
- Scale before regularizing or computing distances.
- Polynomial features let a linear model fit curves — but degree must be
  chosen with care.
- Pipelines keep preprocessing + model in lockstep and survive train/test splits.
- ColumnTransformer routes numeric and categorical columns through
  different transforms in one expression.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Build a `Pipeline` for `tips`: scale numeric columns
(`total_bill`, `size`), one-hot encode `day` and `time`, fit
`LinearRegression` to predict `tip`. Report R².

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import pandas as pd
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import pandas as pd
tips = load_dataset('tips')
pre = ColumnTransformer([
    ('num', StandardScaler(), ['total_bill', 'size']),
    ('cat', OneHotEncoder(drop='first'), ['day', 'time']),
])
pipe = Pipeline([('pre', pre), ('lr', LinearRegression())])
pipe.fit(tips[['total_bill', 'size', 'day', 'time']], tips['tip'])
print(f'R^2 = {pipe.score(tips[["total_bill","size","day","time"]], tips["tip"]):.4f}')


---

## Exercise 2

**Task 2.** Add an interaction `day × time` to the design. Does R²
go up? (Hint: build the dummies first, then multiply.)

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import pandas as pd
X = pd.get_dummies(tips[['day', 'time']], drop_first=True, dtype=float)
for c1 in [c for c in X.columns if c.startswith('day')]:
    for c2 in [c for c in X.columns if c.startswith('time')]:
        X[f'{c1}_x_{c2}'] = X[c1] * X[c2]
X['total_bill'] = tips['total_bill']; X['size'] = tips['size']
m = LinearRegression().fit(X, tips['tip'])
print(f'R^2 with interactions = {m.score(X, tips["tip"]):.4f}')


---

## Exercise 3

**Task 3.** Compare held-out R² before and after the interaction.
Use a fixed `train_test_split(random_state=0, test_size=0.3)`.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
from sklearn.model_selection import train_test_split
# baseline: no interactions
X0 = pd.get_dummies(tips[['day', 'time']], drop_first=True, dtype=float)
X0['total_bill'] = tips['total_bill']; X0['size'] = tips['size']
# with interactions
X1 = X0.copy()
for c1 in [c for c in X0.columns if c.startswith('day')]:
    for c2 in [c for c in X0.columns if c.startswith('time')]:
        X1[f'{c1}_x_{c2}'] = X0[c1] * X0[c2]
y = tips['tip']
X0tr, X0te, ytr, yte = train_test_split(X0, y, test_size=0.3, random_state=0)
X1tr, X1te, _, _ = train_test_split(X1, y, test_size=0.3, random_state=0)
m0 = LinearRegression().fit(X0tr, ytr); m1 = LinearRegression().fit(X1tr, ytr)
print(f'baseline      held-out R^2 = {m0.score(X0te, yte):.4f}')
print(f'+interactions held-out R^2 = {m1.score(X1te, yte):.4f}')
